In [ ]:
import os
import sys
import numpy as np
import tensorflow as tf
from pathlib import Path
from tqdm import tqdm

# Add project root to path and fix working directory
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.chdir(project_root)

from src.data.loader import GliomaDataLoader
from src.models.cvae import CVAE

# GPU Check
print("TensorFlow Version:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

TensorFlow Version: 2.20.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [ ]:
# Setup experiment from config
from src.utils.config_manager import ConfigManager
from src.notebooks.experiment_utils import setup_experiment

config_name = 'exp_102'
config, output_dir = setup_experiment(config_name)

# Load paths from config
DATA_PATH = config['paths']['processed_file']
SPLIT_FILE = config['paths']['splits_file']
SYNTHETIC_OUT_DIR = config['paths']['synthetic_dir']
os.makedirs(SYNTHETIC_OUT_DIR, exist_ok=True)

print(f"✓ Config loaded: {config_name}")
print(f"Data path: {DATA_PATH}")
print(f"Synthetic output: {SYNTHETIC_OUT_DIR}")
print("\nInitializing Loader...")

loader = GliomaDataLoader.from_config(
    h5_data_path=DATA_PATH,
    config=config,
    memory_mapped=False
)

print(f"✓ Loader initialized. Total spectra: {loader.X.shape[0]}")


EXPERIMENT SETUP

Working directory: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data

 Loaded configuration: exp_101
   Description: Complete pipeline: Real data (TRTR) + synthetic (CVAE) + test on synthetic (TSTR) + statistical evaluation

 Output directory: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\exp_101
  Experiment already registered (re-running)

--------------------------------------------------------------------------------
Configuration Summary:
--------------------------------------------------------------------------------
  Preprocessing:
    - AirPLS lambda: 100
    - Signal length: 1738
    - Crop region: None

  CV Strategy:
    - Split source: data/splits/strict_splits.json
    - Use group-aware CV: True
    - Hold out test set: True

  Models:
    - RF n_estimators: 100
    - SVM C: 1.0
    - CVAE epochs: 70


✓ Config loaded: exp_101
Data path: data/processed/glioma_dataset.h5
Synthetic output: data/synthetic

Initializing Loader...
Loa

In [ ]:
# Build CVAE config from centralized exp_102.yaml
cvae_cfg = config['models']['cvae']

cvae_config = {
    'input_shape': (1738, 1),
    'num_classes': 6,
    'latent_dim': cvae_cfg['latent_dim'],
    'loss_weights': {
        'beta': cvae_cfg['beta'],
        'alpha': cvae_cfg['alpha']
    },
    'encoder': {
        'filters': cvae_cfg['encoder']['filters'],
        'kernels': cvae_cfg['encoder']['kernels'],
        'strides': cvae_cfg['encoder']['strides'],
        'use_global_pooling': False,
        'use_skip_gap': False
    },
    'decoder': {
        'initial_filters': cvae_cfg['decoder']['initial_filters'],
        'use_dense_projection': True,
        'filters': cvae_cfg['decoder']['filters'],
        'kernels': cvae_cfg['decoder']['kernels'],
        'strides': cvae_cfg['decoder']['strides']
    }
}

# Training Hyperparameters from config
EPOCHS = cvae_cfg['epochs']
BATCH_SIZE = 1024  # Increased from 256 for CPU efficiency
SAMPLES_PER_CLASS = cvae_cfg['samples_per_class']
LEARNING_RATE = cvae_cfg['learning_rate']
VALIDATION_SPLIT = cvae_cfg['validation_split']

print(f"CVAE Config loaded from exp_102:")
print(f"  - Epochs: {EPOCHS}")
print(f"  - Batch size: {BATCH_SIZE} (optimized)")
print(f"  - Samples/class: {SAMPLES_PER_CLASS}")
print(f"  - Latent dim: {cvae_cfg['latent_dim']}")

CVAE Config loaded from exp_101:
  - Epochs: 70
  - Batch size: 1024 (optimized)
  - Samples/class: 20000
  - Latent dim: 32


In [9]:
import gc

# Setup output directories from config
MODEL_OUT_DIR = config['paths']['models_cvae_dir']
os.makedirs(MODEL_OUT_DIR, exist_ok=True)
os.makedirs(SYNTHETIC_OUT_DIR, exist_ok=True)

print(f"Model weights output: {MODEL_OUT_DIR}")
print(f"Synthetic data output: {SYNTHETIC_OUT_DIR}")
print(f"\n Starting Systematic Synthetic Generation ({SAMPLES_PER_CLASS} per class)")

for fold_data in loader:
    fold_name = fold_data['fold_name']
    
    # Optional: Skip if already generated (useful if it crashes and you restart)
    out_x = f"{SYNTHETIC_OUT_DIR}/{fold_name}_X_synth.npy"
    out_y = f"{SYNTHETIC_OUT_DIR}/{fold_name}_y_synth.npy"
    if os.path.exists(out_x) and os.path.exists(out_y):
        print(f" Skipping {fold_name}, synthetic data already exists.")
        continue

    print(f"\n{'='*60}")
    print(f" Processing {fold_name}")
    print(f"{'='*60}")
    
    # 1. Unpack Data
    X_train, y_train = fold_data['train']
    
    # Fix Labels (0-indexed for Keras categorical)
    if np.min(y_train) == 1:
        y_train = y_train - 1
        
    # Reshape for CNN/CVAE
    X_train = X_train.reshape(-1, 1738, 1)
    
    # One-Hot Encode Labels
    y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes=6)
    
    # 2. Build & Compile CVAE
    tf.keras.backend.clear_session()
    cvae = CVAE(cvae_config)
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    cvae.compile(optimizer=optimizer)
    
    # 3. Train CVAE using custom training loop
    print(f" Training CVAE on {len(X_train)} real samples...")
    print(f"   Epochs={EPOCHS}, Batch={BATCH_SIZE}, Learning_rate={LEARNING_RATE}")
    
    # Create dataset pipeline for training
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train_oh))
    train_dataset = train_dataset.shuffle(buffer_size=len(X_train)).batch(BATCH_SIZE)
    
    # Manual training loop using custom train_step (optimized: no tqdm overhead)
    for epoch in range(EPOCHS):
        total_loss = 0.0
        num_batches = 0
        
        for x_batch, y_batch in train_dataset:
            # Single train_step call - no extra metric conversions
            losses = cvae.train_step((x_batch, y_batch))
            total_loss += float(cvae.total_loss_tracker.result())
            num_batches += 1
        
        # Print summary only once per epoch (not every batch)
        avg_loss = total_loss / num_batches
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} - "
                  f"total_loss: {avg_loss:.4f}, "
                  f"recon: {float(cvae.recon_loss_tracker.result()):.2f}, "
                  f"kl: {float(cvae.kl_loss_tracker.result()):.2f}, "
                  f"class_loss: {float(cvae.class_loss_tracker.result()):.2f}")
    
    print(f"[COMPLETE] Training done for {fold_name}")
    
    # 4. Generate Synthetic Data (using trained weights still in memory - no reload needed)
    print(f"\n[GENERATING] {SAMPLES_PER_CLASS * 6} balanced synthetic spectra for {fold_name}...")
    X_synthetic = []
    y_synthetic = []
    
    # Use batching for generation to avoid OOM with large sample counts
    generation_batch_size = min(5000, SAMPLES_PER_CLASS)  # Generate in chunks of 5k or less
    
    for class_idx in range(6):
        print(f"   Class {class_idx}: ", end='', flush=True)
        class_samples = []
        
        # Generate in batches
        num_generation_batches = (SAMPLES_PER_CLASS + generation_batch_size - 1) // generation_batch_size
        for batch_i in range(num_generation_batches):
            # Calculate batch size (handle last batch if remainder)
            batch_size = min(generation_batch_size, SAMPLES_PER_CLASS - batch_i * generation_batch_size)
            
            # Create pure noise
            random_latent_vectors = tf.random.normal(shape=(batch_size, cvae_config['latent_dim']))
            
            # Create requested class labels
            requested_classes = tf.convert_to_tensor(
                tf.keras.utils.to_categorical([class_idx] * batch_size, num_classes=6),
                dtype=tf.float32
            )
            
            # Decode noise + label into synthetic spectra (using trained decoder in memory)
            generated_spectra = cvae.decoder([random_latent_vectors, requested_classes], training=False)
            class_samples.append(generated_spectra.numpy())
            print(f".", end='', flush=True)
        
        # Concatenate all batches for this class
        X_synthetic.append(np.vstack(class_samples))
        y_synthetic.extend([class_idx] * SAMPLES_PER_CLASS)
        print(f" {SAMPLES_PER_CLASS} samples")
        
    X_synthetic = np.vstack(X_synthetic)
    y_synthetic = np.array(y_synthetic)
    
    # 5. Save CVAE weights and synthetic data
    enc_path = f"{MODEL_OUT_DIR}/encoder_{fold_name}.weights.h5"
    dec_path = f"{MODEL_OUT_DIR}/decoder_{fold_name}.weights.h5"
    
    cvae.encoder.save_weights(enc_path)
    cvae.decoder.save_weights(dec_path)
    
    print(f"[SAVED] Encoder: {enc_path}")
    print(f"[SAVED] Decoder: {dec_path}")
    
    # Save synthetic data
    np.save(out_x, X_synthetic)
    np.save(out_y, y_synthetic)
    print(f"[SAVED] Synthetic data: {out_x}")
    print(f"[SAVED] Synthetic labels: {out_y}")
    print(f"\n[COMPLETE] {fold_name} (training + generation + save)")
    
    # Cleanup for next fold
    del cvae, X_train, y_train_oh, X_synthetic, y_synthetic
    tf.keras.backend.clear_session()
    gc.collect()

print("\n All folds complete! Synthetic datasets ready for TSTR evaluation.")

Model weights output: results/models/cvae
Synthetic data output: data/synthetic

 Starting Systematic Synthetic Generation (20000 per class)

 Using CV splits from: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 131932 train samples, 48118 val samples

 Processing fold_0
 Training CVAE on 131932 real samples...
   Epochs=70, Batch=1024, Learning_rate=0.001
Epoch 1/70 - total_loss: 119.0981, recon: 41.37, kl: 0.89, class_loss: 1.59
Epoch 5/70 - total_loss: 12.6849, recon: 11.09, kl: 0.20, class_loss: 1.23
Epoch 10/70 - total_loss: 7.7738, recon: 7.17, kl: 0.10, class_loss: 0.87
Epoch 15/70 - total_loss: 6.2350, recon: 5.86, kl: 0.07, class_loss: 0.68
Epoch 20/70 - total_loss: 5.4781, recon: 5.20, kl: 0.05, class_loss: 0.55
Epoch 25/70 - total_loss: 5.0265, recon: 4.81, kl: 0.04, class

In [ ]:
%reset -f

## 3. Preprocessing for CVAE
The CVAE expects `(Spectrum, One-Hot-Label)` pairs.

## 5. Diagnostics
Check if the model learned to reconstruct.

## 6. Visual Sanity Check
Reconstruct a random validation sample.